In [ ]:
import requests

OAK = "oak_YOUR_KEY_HERE"

profile = requests.get(f"https://data.ninjakiwi.com/btdb2/users/{OAK}").json()
profile

In [ ]:
matches = requests.get(profile["matches"]).json()
matches

In [ ]:
import requests

OAK = "oak_YOUR_KEY_HERE"

profile = requests.get(f"https://data.ninjakiwi.com/battles2/users/{OAK}").json()
profile

In [ ]:
matches = requests.get(profile["matches"]).json()
matches

In [ ]:
print(list(profile.keys()))

In [ ]:
print(profile["success"], profile["error"])
print(list(profile["body"].keys()))

In [ ]:
matches_resp = requests.get(profile["body"]["matches"]).json()
print(matches_resp["success"], matches_resp["error"])
matches = matches_resp["body"]
matches

In [ ]:
rows = []
for match in matches:
    for side in ["playerLeft", "playerRight"]:
        player = match[side]
        rows.append({
            "match_id": match["id"],
            "side": side,
            "map": match["map"],
            "hero": player["hero"],
            "tower1": player["towerone"],
            "tower2": player["towertwo"],
            "tower3": player["towerthree"],
            "result": player["result"],
            "is_me": player["currentUser"],
            "profile_url": player["profileURL"],
        })

df = pd.DataFrame(rows)
df.to_csv("matches.csv", index=False)
print(df.shape)
df.head()

In [ ]:
import time

seen_match_ids = set(df["match_id"])
all_rows = df.to_dict("records")  # start with what we already have

to_visit = list(df.loc[~df["is_me"], "profile_url"].unique())
visited = set()

MAX_PLAYERS = 20

while to_visit and len(visited) < MAX_PLAYERS:
    url = to_visit.pop(0)
    if url in visited:
        continue
    visited.add(url)

    try:
        profile_resp = requests.get(url).json()
        if not profile_resp["success"]:
            continue
        matches_resp = requests.get(profile_resp["body"]["matches"]).json()
        if not matches_resp["success"]:
            continue
        player_matches = matches_resp["body"]
    except Exception as e:
        print(f"  Skipped one profile ({e})")
        continue

    new_count = 0
    for match in player_matches:
        if match["id"] in seen_match_ids:
            continue
        seen_match_ids.add(match["id"])
        new_count += 1
        for side in ["playerLeft", "playerRight"]:
            player = match[side]
            all_rows.append({
                "match_id": match["id"], "side": side, "map": match["map"],
                "hero": player["hero"], "tower1": player["towerone"],
                "tower2": player["towertwo"], "tower3": player["towerthree"],
                "result": player["result"], "is_me": player["currentUser"],
                "profile_url": player["profileURL"],
            })
            if player["profileURL"] not in visited:
                to_visit.append(player["profileURL"])

    print(f"Visited {len(visited)}/{MAX_PLAYERS} players — +{new_count} new matches (total rows: {len(all_rows)})")
    time.sleep(0.5)

df = pd.DataFrame(all_rows)
df.to_csv("matches.csv", index=False)
print(f"\nDone! {df.shape[0]} rows from {len(visited)} players' match histories")
df.head()

In [ ]:
print(df.shape)
print(df["is_me"].sum(), "rows are you; ", (~df["is_me"]).sum(), "rows are opponents")

In [ ]:
test_url = df.loc[~df["is_me"], "profile_url"].iloc[0]
print("Testing:", test_url)

r = requests.get(test_url)
print("HTTP status code:", r.status_code)
print(r.json())

In [ ]:
test_url = df.loc[~df["is_me"], "profile_url"].iloc[0]
profile_resp = requests.get(test_url).json()
matches_resp = requests.get(profile_resp["body"]["matches"]).json()

opponent_matches = matches_resp["body"]
print("Number of matches this opponent has:", len(opponent_matches))

opponent_match_ids = [m["id"] for m in opponent_matches]
our_match_ids = set(df["match_id"])

print("\nFirst 3 of THEIR match IDs:")
for mid in opponent_match_ids[:3]:
    print(" ", mid)

print("\nFirst 3 of OUR match IDs (for comparison):")
for mid in list(our_match_ids)[:3]:
    print(" ", mid)

overlap = [mid for mid in opponent_match_ids if mid in our_match_ids]
print(f"\n{len(overlap)} of their {len(opponent_match_ids)} matches already exist in our data")

In [ ]:
print("Status code:", requests.get(profile_resp["body"]["matches"]).status_code)
print(matches_resp)

In [ ]:
homs = requests.get("https://data.ninjakiwi.com/battles2/homs").json()
print(homs)

In [ ]:
live_season = next(s for s in homs["body"] if s["live"])
leaderboard = requests.get(live_season["leaderboard"]).json()

print("Players in this HoM leaderboard:", len(leaderboard["body"]))
leaderboard["body"][0]

In [ ]:
import time

df = pd.read_csv("matches.csv")
seen_match_ids = set(df["match_id"])
all_rows = df.to_dict("records")

to_visit = [p["profile"] for p in leaderboard["body"]]  # seed from guaranteed-HoM players
visited = set()

MAX_PLAYERS = 20

while to_visit and len(visited) < MAX_PLAYERS:
    url = to_visit.pop(0)
    if url in visited:
        continue
    visited.add(url)

    profile_resp = requests.get(url).json()
    if not profile_resp["success"]:
        print(f"  Skipped profile — {profile_resp['error']}")
        continue

    matches_resp = requests.get(profile_resp["body"]["matches"]).json()
    if not matches_resp["success"]:
        print(f"  Skipped {profile_resp['body']['displayName']} — {matches_resp['error']}")
        continue

    player_matches = matches_resp["body"]
    new_count = 0
    for match in player_matches:
        if match["id"] in seen_match_ids:
            continue
        seen_match_ids.add(match["id"])
        new_count += 1
        for side in ["playerLeft", "playerRight"]:
            player = match[side]
            all_rows.append({
                "match_id": match["id"], "side": side, "map": match["map"],
                "hero": player["hero"], "tower1": player["towerone"],
                "tower2": player["towertwo"], "tower3": player["towerthree"],
                "result": player["result"], "is_me": player["currentUser"],
                "profile_url": player["profileURL"],
            })
            if player["profileURL"] not in visited and player["profileURL"] not in to_visit:
                to_visit.append(player["profileURL"])

    print(f"Visited {len(visited)}/{MAX_PLAYERS} ({profile_resp['body']['displayName']}) — +{new_count} new matches (total: {len(all_rows)})")
    time.sleep(0.5)

df = pd.DataFrame(all_rows)
df.to_csv("matches.csv", index=False)
print(f"\nDone! {df.shape[0]} rows from {len(visited)} players")
df.head()

In [ ]:
df = pd.read_csv("matches.csv")
df["towers"] = df[["tower1", "tower2", "tower3"]].apply(lambda row: ", ".join(sorted(row)), axis=1)
df["won"] = (df["result"] == "win").astype(int)

In [ ]:
hero_stats = df.groupby("hero")["won"].agg(["mean", "count"]).rename(columns={"mean": "win_rate", "count": "games"})
hero_stats.sort_values("games", ascending=False)

In [ ]:
tower_long = df.melt(id_vars=["won"], value_vars=["tower1", "tower2", "tower3"], value_name="tower")
tower_stats = tower_long.groupby("tower")["won"].agg(["mean", "count"]).rename(columns={"mean": "win_rate", "count": "games"})
tower_stats.sort_values("games", ascending=False)

In [ ]:
combo_stats = df.groupby(["hero", "towers"])["won"].agg(["mean", "count"]).rename(columns={"mean": "win_rate", "count": "games"})
combo_stats.sort_values("games", ascending=False).head(20)

In [ ]:
df["hero_base"] = df["hero"].str.split("_").str[0]

hero_stats_clean = df.groupby("hero_base")["won"].agg(["mean", "count"]).rename(columns={"mean": "win_rate", "count": "games"})
hero_stats_clean.sort_values("games", ascending=False)

In [ ]:
etienne_all = df[df["hero_base"] == "Etienne"]
etienne_excl_me = etienne_all[~etienne_all["is_me"]]

print("Including your own games:", round(etienne_all["won"].mean(), 3), f"({len(etienne_all)} games)")
print("Excluding your own games:", round(etienne_excl_me["won"].mean(), 3), f"({len(etienne_excl_me)} games)")

In [ ]:
df = df.rename(columns={"is_me": "is_profile_owner"})
df.to_csv("matches.csv", index=False)

In [ ]:
import time

df = pd.read_csv("matches.csv")
seen_match_ids = set(df["match_id"])
all_rows = df.to_dict("records")

homs = requests.get("https://data.ninjakiwi.com/battles2/homs").json()
live_season = next(s for s in homs["body"] if s["live"])
leaderboard = requests.get(live_season["leaderboard"]).json()

to_visit = [p["profile"] for p in leaderboard["body"]]
visited = set()

MAX_PLAYERS = 50  # up from 20

while to_visit and len(visited) < MAX_PLAYERS:
    url = to_visit.pop(0)
    if url in visited:
        continue
    visited.add(url)

    profile_resp = requests.get(url).json()
    if not profile_resp["success"]:
        print(f"  Skipped profile — {profile_resp['error']}")
        continue

    matches_resp = requests.get(profile_resp["body"]["matches"]).json()
    if not matches_resp["success"]:
        print(f"  Skipped {profile_resp['body']['displayName']} — {matches_resp['error']}")
        continue

    player_matches = matches_resp["body"]
    new_count = 0
    for match in player_matches:
        if match["id"] in seen_match_ids:
            continue
        seen_match_ids.add(match["id"])
        new_count += 1
        for side in ["playerLeft", "playerRight"]:
            player = match[side]
            all_rows.append({
                "match_id": match["id"], "side": side, "map": match["map"],
                "hero": player["hero"], "tower1": player["towerone"],
                "tower2": player["towertwo"], "tower3": player["towerthree"],
                "result": player["result"], "is_profile_owner": player["currentUser"],
                "profile_url": player["profileURL"],
            })
            if player["profileURL"] not in visited and player["profileURL"] not in to_visit:
                to_visit.append(player["profileURL"])

    print(f"Visited {len(visited)}/{MAX_PLAYERS} ({profile_resp['body']['displayName']}) — +{new_count} new matches (total: {len(all_rows)})")
    time.sleep(0.5)

df = pd.DataFrame(all_rows)
df.to_csv("matches.csv", index=False)
print(f"\nDone! {df.shape[0]} rows from {len(visited)} players")
df.head()

In [ ]:
df["towers"] = df[["tower1", "tower2", "tower3"]].apply(lambda row: ", ".join(sorted(row)), axis=1)
df["won"] = (df["result"] == "win").astype(int)
df["hero_base"] = df["hero"].str.split("_").str[0]
df.to_csv("matches.csv", index=False)

hero_stats_clean = df.groupby("hero_base")["won"].agg(["mean", "count"]).rename(columns={"mean": "win_rate", "count": "games"})
hero_stats_clean.sort_values("games", ascending=False)

In [ ]:
tower_long = df.melt(id_vars=["won"], value_vars=["tower1", "tower2", "tower3"], value_name="tower")
tower_stats = tower_long.groupby("tower")["won"].agg(["mean", "count"]).rename(columns={"mean": "win_rate", "count": "games"})
tower_stats.sort_values("games", ascending=False)

In [ ]:
import time

df = pd.read_csv("matches.csv")
seen_match_ids = set(df["match_id"])
all_rows = df.to_dict("records")

homs = requests.get("https://data.ninjakiwi.com/battles2/homs").json()
live_season = next(s for s in homs["body"] if s["live"])
leaderboard = requests.get(live_season["leaderboard"]).json()

to_visit = [p["profile"] for p in leaderboard["body"]]
visited = set()

MAX_PLAYERS = 90  # up from 50

while to_visit and len(visited) < MAX_PLAYERS:
    url = to_visit.pop(0)
    if url in visited:
        continue
    visited.add(url)

    profile_resp = requests.get(url).json()
    if not profile_resp["success"]:
        print(f"  Skipped profile — {profile_resp['error']}")
        continue

    matches_resp = requests.get(profile_resp["body"]["matches"]).json()
    if not matches_resp["success"]:
        print(f"  Skipped {profile_resp['body']['displayName']} — {matches_resp['error']}")
        continue

    player_matches = matches_resp["body"]
    new_count = 0
    for match in player_matches:
        if match["id"] in seen_match_ids:
            continue
        seen_match_ids.add(match["id"])
        new_count += 1
        for side in ["playerLeft", "playerRight"]:
            player = match[side]
            all_rows.append({
                "match_id": match["id"], "side": side, "map": match["map"],
                "hero": player["hero"], "tower1": player["towerone"],
                "tower2": player["towertwo"], "tower3": player["towerthree"],
                "result": player["result"], "is_profile_owner": player["currentUser"],
                "profile_url": player["profileURL"],
            })
            if player["profileURL"] not in visited and player["profileURL"] not in to_visit:
                to_visit.append(player["profileURL"])

    print(f"Visited {len(visited)}/{MAX_PLAYERS} ({profile_resp['body']['displayName']}) — +{new_count} new matches (total: {len(all_rows)})")
    time.sleep(0.5)

df = pd.DataFrame(all_rows)
df.to_csv("matches.csv", index=False)
print(f"\nDone! {df.shape[0]} rows from {len(visited)} players")
df.head()

In [ ]:
df_rescued = pd.DataFrame(all_rows)
df_rescued.to_csv("matches.csv", index=False)
print(f"Saved {df_rescued.shape[0]} rows safely")

In [ ]:
print(profile_resp)

In [ ]:
import time

df = pd.read_csv("matches.csv")
seen_match_ids = set(df["match_id"])
all_rows = df.to_dict("records")

homs = requests.get("https://data.ninjakiwi.com/battles2/homs").json()
live_season = next(s for s in homs["body"] if s["live"])
leaderboard = requests.get(live_season["leaderboard"]).json()

to_visit = [p["profile"] for p in leaderboard["body"]]
visited = set()

MAX_PLAYERS = 90

def safe_get_json(url, retries=1):
    resp = requests.get(url).json()
    if not resp.get("success") and resp.get("reason") == "Rate limit" and retries > 0:
        print("    Rate limited — waiting 10s and retrying...")
        time.sleep(10)
        return safe_get_json(url, retries=retries - 1)
    return resp

while to_visit and len(visited) < MAX_PLAYERS:
    url = to_visit.pop(0)
    if url in visited:
        continue
    visited.add(url)

    profile_resp = safe_get_json(url)
    if not profile_resp.get("success"):
        reason = profile_resp.get("error") or profile_resp.get("reason") or "unknown error"
        print(f"  Skipped profile — {reason}")
        continue

    matches_resp = safe_get_json(profile_resp["body"]["matches"])
    if not matches_resp.get("success"):
        reason = matches_resp.get("error") or matches_resp.get("reason") or "unknown error"
        print(f"  Skipped {profile_resp['body']['displayName']} — {reason}")
        continue

    player_matches = matches_resp["body"]
    new_count = 0
    for match in player_matches:
        if match["id"] in seen_match_ids:
            continue
        seen_match_ids.add(match["id"])
        new_count += 1
        for side in ["playerLeft", "playerRight"]:
            player = match[side]
            all_rows.append({
                "match_id": match["id"], "side": side, "map": match["map"],
                "hero": player["hero"], "tower1": player["towerone"],
                "tower2": player["towertwo"], "tower3": player["towerthree"],
                "result": player["result"], "is_profile_owner": player["currentUser"],
                "profile_url": player["profileURL"],
            })
            if player["profileURL"] not in visited and player["profileURL"] not in to_visit:
                to_visit.append(player["profileURL"])

    print(f"Visited {len(visited)}/{MAX_PLAYERS} ({profile_resp['body']['displayName']}) — +{new_count} new matches (total: {len(all_rows)})")
    time.sleep(1.5)

df = pd.DataFrame(all_rows)
df.to_csv("matches.csv", index=False)
print(f"\nDone! {df.shape[0]} rows from {len(visited)} players")
df.head()

In [ ]:
import time

df = pd.read_csv("matches.csv")
seen_match_ids = set(df["match_id"])
all_rows = df.to_dict("records")

homs = requests.get("https://data.ninjakiwi.com/battles2/homs").json()
live_season = next(s for s in homs["body"] if s["live"])
leaderboard = requests.get(live_season["leaderboard"]).json()

to_visit = [p["profile"] for p in leaderboard["body"]]
visited = set()

MAX_PLAYERS = 90

def safe_get_json(url, retries=1):
    resp = requests.get(url).json()
    if not resp.get("success") and resp.get("reason") == "Rate limit" and retries > 0:
        print("    Rate limited — waiting 10s and retrying...")
        time.sleep(10)
        return safe_get_json(url, retries=retries - 1)
    return resp

while to_visit and len(visited) < MAX_PLAYERS:
    url = to_visit.pop(0)
    if url in visited:
        continue
    visited.add(url)

    profile_resp = safe_get_json(url)
    if not profile_resp.get("success"):
        reason = profile_resp.get("error") or profile_resp.get("reason") or "unknown error"
        print(f"  Skipped profile — {reason}")
        continue

    matches_resp = safe_get_json(profile_resp["body"]["matches"])
    if not matches_resp.get("success"):
        reason = matches_resp.get("error") or matches_resp.get("reason") or "unknown error"
        print(f"  Skipped {profile_resp['body']['displayName']} — {reason}")
        continue

    player_matches = matches_resp["body"]
    new_count = 0
    for match in player_matches:
        if match["id"] in seen_match_ids:
            continue
        seen_match_ids.add(match["id"])
        new_count += 1
        for side in ["playerLeft", "playerRight"]:
            player = match[side]
            all_rows.append({
                "match_id": match["id"], "side": side, "map": match["map"],
                "hero": player["hero"], "tower1": player["towerone"],
                "tower2": player["towertwo"], "tower3": player["towerthree"],
                "result": player["result"], "is_profile_owner": player["currentUser"],
                "profile_url": player["profileURL"],
            })
            if player["profileURL"] not in visited and player["profileURL"] not in to_visit:
                to_visit.append(player["profileURL"])

    print(f"Visited {len(visited)}/{MAX_PLAYERS} ({profile_resp['body']['displayName']}) — +{new_count} new matches (total: {len(all_rows)})")

    if len(visited) % 10 == 0:
        pd.DataFrame(all_rows).to_csv("matches.csv", index=False)
        print("  [checkpoint saved]")

    time.sleep(1.5)

df = pd.DataFrame(all_rows)
df.to_csv("matches.csv", index=False)
print(f"\nDone! {df.shape[0]} rows from {len(visited)} players")
df.head()

In [ ]:
import time

df = pd.read_csv("matches.csv")
seen_match_ids = set(df["match_id"])
all_rows = df.to_dict("records")

homs = requests.get("https://data.ninjakiwi.com/battles2/homs").json()
live_season = next(s for s in homs["body"] if s["live"])
leaderboard = requests.get(live_season["leaderboard"]).json()

to_visit = [p["profile"] for p in leaderboard["body"]]
visited = set()

MAX_PLAYERS = 150

def safe_get_json(url, retries=1):
    resp = requests.get(url).json()
    if not resp.get("success") and resp.get("reason") == "Rate limit" and retries > 0:
        print("    Rate limited — waiting 10s and retrying...")
        time.sleep(10)
        return safe_get_json(url, retries=retries - 1)
    return resp

while to_visit and len(visited) < MAX_PLAYERS:
    url = to_visit.pop(0)
    if url in visited:
        continue
    visited.add(url)

    profile_resp = safe_get_json(url)
    if not profile_resp.get("success"):
        reason = profile_resp.get("error") or profile_resp.get("reason") or "unknown error"
        print(f"  Skipped profile — {reason}")
        continue

    matches_resp = safe_get_json(profile_resp["body"]["matches"])
    if not matches_resp.get("success"):
        reason = matches_resp.get("error") or matches_resp.get("reason") or "unknown error"
        print(f"  Skipped {profile_resp['body']['displayName']} — {reason}")
        continue

    player_matches = matches_resp["body"]
    new_count = 0
    for match in player_matches:
        is_new_match = match["id"] not in seen_match_ids
        if is_new_match:
            seen_match_ids.add(match["id"])
            new_count += 1

        for side in ["playerLeft", "playerRight"]:
            player = match[side]
            if is_new_match:
                all_rows.append({
                    "match_id": match["id"], "side": side, "map": match["map"],
                    "hero": player["hero"], "tower1": player["towerone"],
                    "tower2": player["towertwo"], "tower3": player["towerthree"],
                    "result": player["result"], "is_profile_owner": player["currentUser"],
                    "profile_url": player["profileURL"],
                })
            # explore this opponent regardless of whether the match itself was new
            if player["profileURL"] not in visited and player["profileURL"] not in to_visit:
                to_visit.append(player["profileURL"])

    print(f"Visited {len(visited)}/{MAX_PLAYERS} ({profile_resp['body']['displayName']}) — +{new_count} new matches (total: {len(all_rows)})")

    if len(visited) % 10 == 0:
        pd.DataFrame(all_rows).to_csv("matches.csv", index=False)
        print("  [checkpoint saved]")

    time.sleep(1.5)

df = pd.DataFrame(all_rows)
df.to_csv("matches.csv", index=False)
print(f"\nDone! {df.shape[0]} rows from {len(visited)} players")
df.head()

In [ ]:
df = pd.read_csv("matches.csv")
df["towers"] = df[["tower1", "tower2", "tower3"]].apply(lambda row: ", ".join(sorted(row)), axis=1)
df["won"] = (df["result"] == "win").astype(int)
df["hero_base"] = df["hero"].str.split("_").str[0]
df.to_csv("matches.csv", index=False)
print(df.shape)